# Agent 数据结构速查手册

> 手敲 s01~s20 的代码时，忘了 `messages` / `response` / `block` 长什么样，回这里看。

覆盖内容：

1. `messages` —— 你发给模型的对话历史
2. `response` —— 模型返回的对象
3. `content block` —— 一条消息由多个「块」拼成（text / tool_use / tool_result ...）
4. 一轮完整的工具调用循环里，messages 是怎么增长的
5. Anthropic (Claude) vs OpenAI 结构对照
6. 流式输出的事件序列

**大部分单元格用纯 dict 演示，不需要 API key 就能跑。** 最后一个单元格才会真正调用 API（可选）。

## 0. 三层结构总览

```
messages  = [ message, message, message, ... ]   ← 一整段对话历史（list）
              │
              └─ message = { "role": ..., "content": ... }   ← 一条消息（dict）
                              │
                              └─ content = [ block, block, ... ]  ← 由多个内容块组成
                                            │
                                            └─ block = { "type": "text" / "tool_use" / ... }
```

记住三层：**messages（对话） > message（一条消息） > block（消息里的一块）**。

## 1. messages —— 你发给模型的东西

`messages` 是一个 list，每个元素是一条消息，只有两个字段：`role` 和 `content`。

- `role`: `"user"` 或 `"assistant"`（system 一般单独传，不放在 messages 里）
- `content`: 可以是**一段字符串**（简单情况），也可以是**一个 block 列表**（有工具/图片时）

In [ ]:
# 最简单的 messages：content 就是字符串
messages = [
    {"role": "user", "content": "读一下 a.txt"},
]

print(type(messages))          # list
print(type(messages[0]))       # dict
print(messages[0]["role"])     # user
print(messages[0]["content"])  # 读一下 a.txt

## 2. response —— 模型返回的东西

调用 `client.messages.create(...)` 后拿到的 `response` 对象，关键字段：

| 字段 | 含义 |
|------|------|
| `response.content` | **block 列表**，回复的正文（可能同时有文字块 + 工具块） |
| `response.stop_reason` | 停止原因：`"end_turn"`（说完了）/ `"tool_use"`（要调工具）/ `"max_tokens"` |
| `response.role` | 永远是 `"assistant"` |
| `response.usage` | token 用量 |

主循环里最关键的判断就是 `if response.stop_reason == "tool_use"`。

下面用一个**模拟对象**还原真实 response 的结构（真实 SDK 返回的是对象，用 `.` 访问；这里用一个小类模拟 `.` 访问）。

In [ ]:
# 用一个小工具把 dict 变成可以用 . 访问的对象，模拟真实 SDK 返回值
class Obj:
    """把嵌套 dict/list 变成可以 obj.field 访问的对象（仅用于演示）。"""
    def __init__(self, d):
        for k, v in d.items():
            setattr(self, k, self._wrap(v))
    @classmethod
    def _wrap(cls, v):
        if isinstance(v, dict):
            return cls(v)
        if isinstance(v, list):
            return [cls._wrap(x) for x in v]
        return v
    def __repr__(self):
        return f"Obj({self.__dict__})"

# 模拟：模型只回了一段文字（不调工具）
response = Obj({
    "role": "assistant",
    "stop_reason": "end_turn",
    "content": [
        {"type": "text", "text": "a.txt 的内容是 hello world"},
    ],
})

print("stop_reason:", response.stop_reason)
print("content 是个 list:", isinstance(response.content, list))
print("第 0 块类型:", response.content[0].type)
print("第 0 块文字:", response.content[0].text)

## 3. content block —— 一条消息由多个「块」拼成

这是 Anthropic API 的核心设计：**一条消息的内容是一个异构块数组**，不是一段纯文本。

常见 block 类型：

| block.type | 关键字段 | 谁产生的 |
|-----------|---------|---------|
| `"text"` | `.text` | 模型 |
| `"tool_use"` | `.id` `.name` `.input`(dict) | 模型（它想调工具） |
| `"tool_result"` | `.tool_use_id` `.content` | **你**（把工具执行结果回传给模型） |

下面这个 response 同时包含**文字块 + 工具调用块**——这就是为什么主循环要 `for block in response.content` 逐块遍历、还要先判 `block.type`。

In [ ]:
# 模拟：模型说了句话，然后决定调用 read_file 工具
response = Obj({
    "role": "assistant",
    "stop_reason": "tool_use",          # ← 注意：要调工具了
    "content": [
        {"type": "text", "text": "我来读一下这个文件"},
        {"type": "tool_use",
         "id": "toolu_01ABC",           # 这次调用的唯一 id，回传结果时要对上
         "name": "read_file",           # 工具名 → 决定调 TOOL_HANDLERS 里哪个函数
         "input": {"path": "a.txt"}},   # 参数，已经是解析好的 dict
    ],
})

# 主循环里的遍历逻辑，原样搬过来
for block in response.content:
    print("—", block.type)
    if block.type == "text":
        print("   文字:", block.text)
    elif block.type == "tool_use":
        print("   id   :", block.id)
        print("   name :", block.name)
        print("   input:", block.input, "  <- 是 dict，可以 **block.input 解包")

In [ ]:
# 你回传结果时，构造的是 tool_result block，放进一条 role=user 的消息里
# 关键：tool_use_id 必须等于上面那个 tool_use 的 id，模型才知道这是哪次调用的结果
tool_result_message = {
    "role": "user",
    "content": [
        {"type": "tool_result",
         "tool_use_id": "toolu_01ABC",   # ← 对应上面的 block.id
         "content": "hello world"},        # 工具执行的输出
    ],
}
print(tool_result_message)

## 4. 一轮工具调用循环里，messages 是怎么增长的

这是理解 agent loop 的关键。一次「读文件」任务，messages 会经历 4 个阶段：

```
① user 提问
② assistant 回复（含 tool_use 块）      ← append(response.content)
③ user 回传 tool_result                 ← append 工具结果
④ assistant 最终回答（stop_reason=end_turn，循环结束）
```

注意②和③：模型的工具调用是 assistant 消息，你回传的结果是 user 消息——**工具结果在协议上算「用户」发的**。

In [ ]:
# 完整演示一轮循环后 messages 的样子（用 dict 表示，便于打印）
messages = []

# ① 用户提问
messages.append({"role": "user", "content": "读一下 a.txt"})

# ② 模型回复：要调工具（真实代码里是 messages.append({"role":"assistant","content":response.content}))
messages.append({"role": "assistant", "content": [
    {"type": "text", "text": "我来读一下"},
    {"type": "tool_use", "id": "toolu_01ABC", "name": "read_file", "input": {"path": "a.txt"}},
]})

# ③ 你执行工具，把结果作为 user 消息回传
messages.append({"role": "user", "content": [
    {"type": "tool_result", "tool_use_id": "toolu_01ABC", "content": "hello world"},
]})

# ④ 模型看到结果后给出最终回答
messages.append({"role": "assistant", "content": [
    {"type": "text", "text": "a.txt 的内容是 hello world"},
]})

import json
for i, m in enumerate(messages):
    print(f"[{i}] role={m['role']}")
    print("    ", json.dumps(m["content"], ensure_ascii=False))

## 5. Anthropic (Claude) vs OpenAI —— 结构对照

两家切分工具调用的方式不同，这是最容易混的地方。

| | Anthropic | OpenAI |
|---|---|---|
| 工具调用放在哪 | `content` 里的一个 **block** (`type=tool_use`) | message 上单独的 `tool_calls` 字段（和 content 平级） |
| 参数格式 | `input` 是**已解析的 dict** | `arguments` 是 **JSON 字符串**，要自己 `json.loads()` |
| 文字和工具 | 同一个 content 数组里混着 | 分在 `content` 和 `tool_calls` 两处 |
| 结果怎么回传 | `role=user` + `tool_result` block | `role=tool` 的独立消息 + `tool_call_id` |

对应关系：**Anthropic 的 `tool_use` block ≈ OpenAI 的 `tool_call`**（不是 message，block 比 message 小一层）。

In [ ]:
import json

# ---- Anthropic 风格：工具调用是 content 里的一个 block，input 是 dict ----
anthropic_msg = {
    "role": "assistant",
    "content": [
        {"type": "text", "text": "我来读文件"},
        {"type": "tool_use", "id": "toolu_01", "name": "read_file",
         "input": {"path": "a.txt"}},          # dict，可直接 **input
    ],
}

# ---- OpenAI 风格：工具调用在 tool_calls 字段，arguments 是字符串 ----
openai_msg = {
    "role": "assistant",
    "content": None,                              # 文字在这（这里为空）
    "tool_calls": [
        {"id": "call_01", "type": "function",
         "function": {"name": "read_file",
                      "arguments": '{"path": "a.txt"}'}},  # ← 字符串！
    ],
}

print("Anthropic input 类型:", type(anthropic_msg["content"][1]["input"]))
print("OpenAI  arguments 类型:", type(openai_msg["tool_calls"][0]["function"]["arguments"]))

# OpenAI 要多一步：把字符串解析成 dict
openai_args = json.loads(openai_msg["tool_calls"][0]["function"]["arguments"])
print("OpenAI 解析后:", openai_args, type(openai_args))

In [ ]:
# 结果回传方式也不同

# Anthropic：role=user，装一个 tool_result block
anthropic_result = {
    "role": "user",
    "content": [
        {"type": "tool_result", "tool_use_id": "toolu_01", "content": "hello world"},
    ],
}

# OpenAI：一条独立的 role=tool 消息，用 tool_call_id 对应
openai_result = {
    "role": "tool",
    "tool_call_id": "call_01",
    "content": "hello world",
}

print("Anthropic:", anthropic_result)
print("OpenAI   :", openai_result)

## 6. 流式输出 —— 事件序列

非流式：一次性拿到完整的 `response.content`（所有 block 都齐了）。

流式：服务器一小片一小片吐，事件**全都围绕 content_block 命名**——因为 API 把回复切成 block，流式时才能告诉你「现在是第几块、什么类型、增量内容是什么」。

**非流式 = 流式的「攒齐版」**：SDK 底层把所有 delta 按 block 拼好，最后交给你完整的 `response.content`。

下面模拟一次流式事件序列（文字块 + 工具块）。

In [ ]:
# 模拟流式事件流（真实事件字段更多，这里保留核心）
stream_events = [
    {"type": "message_start"},
    {"type": "content_block_start", "index": 0, "block_type": "text"},
    {"type": "content_block_delta", "index": 0, "text": "我来"},
    {"type": "content_block_delta", "index": 0, "text": "帮你读"},
    {"type": "content_block_stop", "index": 0},
    {"type": "content_block_start", "index": 1, "block_type": "tool_use", "name": "read_file"},
    {"type": "content_block_delta", "index": 1, "partial_json": '{"path"'},
    {"type": "content_block_delta", "index": 1, "partial_json": ': "a.txt"}'},
    {"type": "content_block_stop", "index": 1},
    {"type": "message_stop"},
]

# 把流式事件「攒」回成非流式的 content 列表——这就是 SDK 底层做的事
blocks = {}
for ev in stream_events:
    t = ev["type"]
    if t == "content_block_start":
        blocks[ev["index"]] = {"type": ev["block_type"], "buf": ""}
        if ev["block_type"] == "tool_use":
            blocks[ev["index"]]["name"] = ev["name"]
    elif t == "content_block_delta":
        piece = ev.get("text") or ev.get("partial_json") or ""
        blocks[ev["index"]]["buf"] += piece
    print(f"{t:22} {ev.get('index','')} {ev.get('text') or ev.get('partial_json') or ''}")

print("\n攒齐后的 blocks:")
import json
for i, b in blocks.items():
    print(f"  [{i}] type={b['type']:9} 内容={b['buf']!r}", f"name={b.get('name')}" if 'name' in b else "")
# 注意：工具块的 partial_json 逐片拼成完整 JSON 字符串后，再 json.loads 得到 input dict

## 7.（可选）真实调用一次，亲眼看看 response

下面才会真正打 API。需要 `.env` 里配好 `ANTHROPIC_BASE_URL` / `MODEL_ID` / key（和 s01~s20 用的一致）。

不想联网就跳过这格，前面 6 节已经把结构讲全了。

In [ ]:
import os
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv(override=True)
if os.getenv("ANTHROPIC_BASE_URL"):
    os.environ.pop("ANTHROPIC_AUTH_TOKEN", None)

client = Anthropic(base_url=os.getenv("ANTHROPIC_BASE_URL"))
MODEL = os.environ["MODEL_ID"]

resp = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    messages=[{"role": "user", "content": "用一句话介绍你自己"}],
)

print("stop_reason:", resp.stop_reason)
print("content 长度:", len(resp.content))
for block in resp.content:
    print("block.type =", block.type)
    if block.type == "text":
        print("  text =", block.text)
print("\n原始对象类型:", type(resp))
print("content[0] 类型:", type(resp.content[0]))

In [ ]:
# 再看一次「带工具」的真实 response——模型会返回 stop_reason=tool_use
TOOLS = [{
    "name": "read_file",
    "description": "Read file contents.",
    "input_schema": {
        "type": "object",
        "properties": {"path": {"type": "string"}},
        "required": ["path"],
    },
}]

resp2 = client.messages.create(
    model=MODEL,
    max_tokens=1000,
    tools=TOOLS,
    messages=[{"role": "user", "content": "读一下 hello.py"}],
)

print("stop_reason:", resp2.stop_reason)
for block in resp2.content:
    print("—", block.type)
    if block.type == "text":
        print("   text  =", block.text)
    elif block.type == "tool_use":
        print("   id    =", block.id)
        print("   name  =", block.name)
        print("   input =", block.input, type(block.input))